In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import io

In [2]:
!pip install mlflow albumentations -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 110.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 127.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import mlflow
import mlflow.pytorch

In [4]:
!git clone https://github.com/2245-RN-ITBA/skin-dataset-classification.git
import os
os.chdir("skin-dataset-classification")

Cloning into 'skin-dataset-classification'...
remote: Enumerating objects: 934, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 934 (delta 17), reused 15 (delta 15), pack-reused 911 (from 2)
Receiving objects: 100% (934/934), 222.43 MiB | 23.68 MiB/s, done.
Resolving deltas: 100% (31/31), done.


In [5]:
mlflow.set_tracking_uri("sqlite:///mlflow_1.db")
mlflow.set_experiment("MLP_Clasificador_Imagenes")

2026/06/20 17:11:26 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/20 17:11:26 INFO mlflow.store.db.utils: Updating database tables
2026/06/20 17:11:31 INFO mlflow.tracking.fluent: Experiment with name 'MLP_Clasificador_Imagenes' does not exist. Creating a new experiment.


<Experiment: artifact_location='/content/skin-dataset-classification/mlruns/1', creation_time=1781975491691, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781975491691, lifecycle_stage='active', name='MLP_Clasificador_Imagenes', tags={}, trace_location=None, workspace='default'>

In [6]:
from torch.utils.tensorboard import SummaryWriter
import torchvision.utils as vutils

In [7]:
# Función para loguear una figura matplotlib en TensorBoard
def plot_to_tensorboard(fig, writer, tag, step):
    buf = io.BytesIO()
    fig.savefig(buf, format='png')
    buf.seek(0)
    image = Image.open(buf).convert("RGB")
    image = np.array(image)
    image = torch.tensor(image).permute(2, 0, 1) / 255.0
    writer.add_image(tag, image, global_step=step)
    plt.close(fig)


In [8]:
# Función para matriz de confusión y clasificación
def log_classification_report(model, loader, writer, step, prefix="val"):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    fig_cm, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=train_dataset.label_encoder.classes_)
    disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
    ax.set_title(f'{prefix.title()} - Confusion Matrix')

    # Guardar localmente y subir a MLflow
    fig_path = f"confusion_matrix_{prefix}_epoch_{step}.png"
    fig_cm.savefig(fig_path)
    mlflow.log_artifact(fig_path)
    os.remove(fig_path)

    plot_to_tensorboard(fig_cm, writer, f"{prefix}/confusion_matrix", step)

    cls_report = classification_report(all_labels, all_preds, target_names=train_dataset.label_encoder.classes_)
    writer.add_text(f"{prefix}/classification_report", f"<pre>{cls_report}</pre>", step)

    # También loguear texto del reporte
    with open(f"classification_report_{prefix}_epoch_{step}.txt", "w") as f:
        f.write(cls_report)
    mlflow.log_artifact(f.name)
    os.remove(f.name)


In [9]:
# Crear directorio de logs
log_dir = "runs/mlp_experimento_1"
writer = SummaryWriter(log_dir=log_dir)

In [10]:
class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.image_paths = []
        self.labels = []

        class_names = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}

        for cls in class_names:
            cls_dir = os.path.join(root_dir, cls)
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                    self.image_paths.append(os.path.join(cls_dir, fname))
                    self.labels.append(cls)

        self.label_encoder = LabelEncoder()
        self.labels = self.label_encoder.fit_transform(self.labels)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = np.array(Image.open(self.image_paths[idx]).convert("RGB"))
        label = self.labels[idx]

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented["image"]

        return image, label


In [11]:
train_transform = A.Compose([
    A.Resize(64, 64),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(),
    ToTensorV2()
])


In [12]:
val_test_transform = A.Compose([
    A.Resize(64, 64),
    A.Normalize(),
    ToTensorV2()
])


In [13]:
# Paths
train_dir = "data/Split_smol/train"
val_dir = "data/Split_smol/val/"

In [14]:
train_dataset = CustomImageDataset(train_dir, transform=train_transform)
val_dataset   = CustomImageDataset(val_dir, transform=val_test_transform)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size)


In [15]:
# modificada

class MLPClassifier(nn.Module):
    def __init__(self,
                 input_size=64*64*3,
                 num_classes=10,
                 dropout_p=0.0,
                 use_batchnorm=False,
                 init_mode='default'):
        super().__init__()

        layers = [nn.Flatten()]

        layers.append(nn.Linear(input_size, 512))
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(512))
        layers.append(nn.ReLU())
        if dropout_p > 0.0:
            layers.append(nn.Dropout(p=dropout_p))

        layers.append(nn.Linear(512, 128))
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(128))
        layers.append(nn.ReLU())
        if dropout_p > 0.0:
            layers.append(nn.Dropout(p=dropout_p))

        layers.append(nn.Linear(128, num_classes))

        self.model = nn.Sequential(*layers)

        if init_mode != 'default':
            self.init_weights(init_mode)

    def init_weights(self, mode='he'):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                if mode == 'he':
                    nn.init.kaiming_normal_(m.weight)
                elif mode == 'xavier':
                    nn.init.xavier_uniform_(m.weight)
                elif mode == 'uniform':
                    nn.init.uniform_(m.weight, -0.1, 0.1)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.model(x)

In [16]:
# modificada

device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(set(train_dataset.labels))

model = MLPClassifier(
    num_classes=num_classes,
    dropout_p=0.5,          # 0.0 = sin Dropout
    use_batchnorm=True,     # False = sin BatchNorm
    init_mode='he'
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

print(f"Modelo en: {device}")
print(f"Parámetros entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Modelo en: cuda
Parámetros entrenables: 6,360,073


In [17]:
# Entrenamiento y validación
def evaluate(model, loader, epoch=None, prefix="val"):
    log_classification_report(model, val_loader, writer, step=epoch, prefix="val")
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for i, (images, labels) in enumerate(loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            loss_sum += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            # Loguear imágenes del primer batch
            if i == 0 and epoch is not None:
                img_grid = vutils.make_grid(images[:8].cpu(), normalize=True)
                writer.add_image(f"{prefix}/images", img_grid, global_step=epoch)

    acc = 100.0 * correct / total
    avg_loss = loss_sum / len(loader)

    if epoch is not None:
        writer.add_scalar(f"{prefix}/loss", avg_loss, epoch)
        writer.add_scalar(f"{prefix}/accuracy", acc, epoch)

    return avg_loss, acc

In [18]:
# modificada

# Loop de entrenamiento
n_epochs = 10
with mlflow.start_run():
    # Log hiperparámetros
    mlflow.log_params({
        "model": "MLPClassifier",
        "input_size": 64*64*3,
        "batch_size": batch_size,
        "lr": 1e-3,
        "epochs": n_epochs,
        "optimizer": "Adam",
        "loss_fn": "CrossEntropyLoss",
        "train_dir": train_dir,
        "val_dir": val_dir,
    })
    for epoch in range(n_epochs):
        model.train()
        running_loss = 0.0
        correct, total = 0, 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs}"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = 100.0 * correct / total
        val_loss, val_acc = evaluate(model, val_loader, epoch=epoch, prefix="val")

        print(f"Epoch {epoch+1}:")
        print(f"  Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%")
        print(f"  Val   Loss: {val_loss:.4f}, Accuracy: {val_acc:.2f}%")

        writer.add_scalar("train/loss", train_loss, epoch)
        writer.add_scalar("train/accuracy", train_acc, epoch)


    #  Histogramas de pesos
        for name, param in model.named_parameters():
            writer.add_histogram(name, param, epoch)


        # Log en MLflow
        mlflow.log_metrics({
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc
        }, step=epoch)

        # Guardar modelo
    torch.save(model.state_dict(), "mlp_model.pth")
    print("Modelo guardado como 'mlp_model.pth'")
    mlflow.log_artifact("mlp_model.pth")
    print("Modelo guardado como 'mlp_model.pth'")

Epoch 1/10: 100%|██████████| 22/22 [00:05<00:00,  4.15it/s]


Epoch 1:
  Train Loss: 2.2156, Accuracy: 26.97%
  Val   Loss: 1.6979, Accuracy: 40.88%


Epoch 2/10: 100%|██████████| 22/22 [00:05<00:00,  4.19it/s]


Epoch 2:
  Train Loss: 1.8815, Accuracy: 32.71%
  Val   Loss: 1.5604, Accuracy: 39.78%


Epoch 3/10: 100%|██████████| 22/22 [00:04<00:00,  5.25it/s]


...

Epoch 8:
  Train Loss: 1.4335, Accuracy: 46.05%
  Val   Loss: 1.2037, Accuracy: 54.70%


Epoch 9/10: 100%|██████████| 22/22 [00:04<00:00,  5.01it/s]


Epoch 9:
  Train Loss: 1.3212, Accuracy: 49.50%
  Val   Loss: 1.2028, Accuracy: 54.70%


Epoch 10/10: 100%|██████████| 22/22 [00:04<00:00,  4.45it/s]


Epoch 10:
  Train Loss: 1.2942, Accuracy: 50.79%
  Val   Loss: 1.1521, Accuracy: 57.46%
Modelo guardado como 'mlp_model.pth'
Modelo guardado como 'mlp_model.pth'


In [19]:
reg_configs = [
    {"run_name": "baseline",             "dropout_p": 0.0, "use_batchnorm": False, "weight_decay": 0.0,  "init_mode": "default"},
    {"run_name": "dropout_only",         "dropout_p": 0.5, "use_batchnorm": False, "weight_decay": 0.0,  "init_mode": "default"},
    {"run_name": "batchnorm_dropout",    "dropout_p": 0.5, "use_batchnorm": True,  "weight_decay": 0.0,  "init_mode": "default"},
    {"run_name": "batchnorm_dropout_wd", "dropout_p": 0.5, "use_batchnorm": True,  "weight_decay": 1e-4, "init_mode": "he"},
]

N_EPOCHS_REG = 15

for cfg in reg_configs:
    print(f"\n=== Entrenando: {cfg['run_name']} ===")

    t_dataset = CustomImageDataset(train_dir, transform=train_transform)
    v_dataset = CustomImageDataset(val_dir,   transform=val_test_transform)
    t_loader  = DataLoader(t_dataset, batch_size=32, shuffle=True)
    v_loader  = DataLoader(v_dataset, batch_size=32)

    m = MLPClassifier(
        num_classes=len(set(t_dataset.labels)),
        dropout_p=cfg["dropout_p"],
        use_batchnorm=cfg["use_batchnorm"],
        init_mode=cfg["init_mode"]
    ).to(device)

    crit = nn.CrossEntropyLoss()
    opt  = optim.Adam(m.parameters(), lr=1e-3, weight_decay=cfg["weight_decay"])
    w = SummaryWriter(log_dir=f"runs/reg_{cfg['run_name']}")

    with mlflow.start_run(run_name=cfg["run_name"]):
        mlflow.log_params({**cfg, "epochs": N_EPOCHS_REG, "lr": 1e-3, "batch_size": 32})

        best_val_acc = 0
        patience, wait = 5, 0

        for epoch in range(N_EPOCHS_REG):
            m.train()
            running_loss, correct, total = 0.0, 0, 0
            for imgs, lbls in t_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                opt.zero_grad()
                out  = m(imgs)
                loss = crit(out, lbls)
                loss.backward()
                opt.step()
                running_loss += loss.item()
                _, p = torch.max(out, 1)
                correct += (p == lbls).sum().item()
                total   += lbls.size(0)

            t_loss = running_loss / len(t_loader)
            t_acc  = 100.0 * correct / total

            m.eval()
            v_loss_sum, v_correct, v_total = 0.0, 0, 0
            with torch.no_grad():
                for imgs, lbls in v_loader:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    out  = m(imgs)
                    loss = crit(out, lbls)
                    _, p = torch.max(out, 1)
                    v_loss_sum += loss.item()
                    v_correct  += (p == lbls).sum().item()
                    v_total    += lbls.size(0)

            v_loss = v_loss_sum / len(v_loader)
            v_acc  = 100.0 * v_correct / v_total

            w.add_scalar("train/loss",     t_loss, epoch)
            w.add_scalar("train/accuracy", t_acc,  epoch)
            w.add_scalar("val/loss",       v_loss, epoch)
            w.add_scalar("val/accuracy",   v_acc,  epoch)

            for name, param in m.named_parameters():
                w.add_histogram(name, param, epoch)

            mlflow.log_metrics({
                "train_loss": t_loss, "train_accuracy": t_acc,
                "val_loss":   v_loss, "val_accuracy":   v_acc,
            }, step=epoch)

            print(f"  Epoch {epoch+1:2d}: train_acc={t_acc:.1f}%  val_acc={v_acc:.1f}%")

            if v_acc > best_val_acc:
                best_val_acc = v_acc
                wait = 0
                torch.save(m.state_dict(), f"best_{cfg['run_name']}.pth")
            else:
                wait += 1
                if wait >= patience:
                    print(f"  Early stopping en epoch {epoch+1}")
                    break

        mlflow.log_metric("best_val_accuracy", best_val_acc)
        mlflow.log_artifact(f"best_{cfg['run_name']}.pth")

    w.close()
    print(f"  Mejor val_acc: {best_val_acc:.2f}%")


=== Entrenando: baseline ===
  Epoch  1: train_acc=28.8%  val_acc=37.0%
  Epoch  2: train_acc=47.2%  val_acc=44.2%
  Epoch  3: train_acc=48.1%  val_acc=51.4%
  Epoch  4: train_acc=54.1%  val_acc=47.5%
  Epoch  5: train_acc=58.1%  val_acc=49.2%
  Epoch  6: train_acc=59.4%  val_acc=56.9%
  Epoch  7: train_acc=56.8%  val_acc=55.8%
  Epoch  8: train_acc=61.5%  val_acc=51.9%
  Epoch  9: train_acc=65.3%  val_acc=55.2%
  Epoch 10: train_acc=66.1%  val_acc=53.0%
  Epoch 11: train_acc=65.6%  val_acc=52.5%
  Early stopping en epoch 11
  Mejor val_acc: 56.91%

=== Entrenando: dropout_only ===
  Epoch  1: train_acc=18.7%  val_acc=30.4%
  Epoch  2: train_acc=20.9%  val_acc=29.3%
  Epoch  3: train_acc=25.3%  val_acc=40.3%
  Epoch  4: train_acc=32.0%  val_acc=42.0%
  Epoch  5: train_acc=36.3%  val_acc=39.2%
  Epoch  6: train_acc=34.3%  val_acc=36.5%
  Epoch  7: train_acc=37.7%  val_acc=47.5%
  Epoch  8: train_acc=35.7%  val_acc=40.9%
  Epoch  9: train_acc=35.6%  val_acc=44.8%
  Epoch 10: train_acc=3

In [20]:
init_configs = ['he', 'xavier', 'uniform', 'default']
N_EPOCHS_INIT = 10

for init_mode in init_configs:
    print(f"\n=== Inicialización: {init_mode} ===")

    t_dataset = CustomImageDataset(train_dir, transform=train_transform)
    v_dataset = CustomImageDataset(val_dir,   transform=val_test_transform)
    t_loader  = DataLoader(t_dataset, batch_size=32, shuffle=True)
    v_loader  = DataLoader(v_dataset, batch_size=32)

    m = MLPClassifier(
        num_classes=len(set(t_dataset.labels)),
        dropout_p=0.5,
        use_batchnorm=True,
        init_mode=init_mode
    ).to(device)

    crit = nn.CrossEntropyLoss()
    opt  = optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-4)
    w = SummaryWriter(log_dir=f"runs/init_{init_mode}")

    with mlflow.start_run(run_name=f"init_{init_mode}"):
        mlflow.log_params({
            "init_mode": init_mode, "dropout_p": 0.5,
            "use_batchnorm": True, "weight_decay": 1e-4, "epochs": N_EPOCHS_INIT,
        })

        for name, param in m.named_parameters():
            w.add_histogram(f"init_{name}", param, 0)

        best_val_acc = 0
        for epoch in range(N_EPOCHS_INIT):
            m.train()
            running_loss, correct, total = 0.0, 0, 0
            for imgs, lbls in t_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                opt.zero_grad()
                out  = m(imgs)
                loss = crit(out, lbls)
                loss.backward()
                opt.step()
                running_loss += loss.item()
                _, p = torch.max(out, 1)
                correct += (p == lbls).sum().item()
                total   += lbls.size(0)

            t_loss = running_loss / len(t_loader)
            t_acc  = 100.0 * correct / total

            m.eval()
            v_loss_sum, v_correct, v_total = 0.0, 0, 0
            with torch.no_grad():
                for imgs, lbls in v_loader:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    out  = m(imgs)
                    loss = crit(out, lbls)
                    _, p = torch.max(out, 1)
                    v_loss_sum += loss.item()
                    v_correct  += (p == lbls).sum().item()
                    v_total    += lbls.size(0)

            v_loss = v_loss_sum / len(v_loader)
            v_acc  = 100.0 * v_correct / v_total

            w.add_scalar("train/loss",     t_loss, epoch)
            w.add_scalar("train/accuracy", t_acc,  epoch)
            w.add_scalar("val/loss",       v_loss, epoch)
            w.add_scalar("val/accuracy",   v_acc,  epoch)

            for name, param in m.named_parameters():
                w.add_histogram(name, param, epoch)

            mlflow.log_metrics({
                "train_loss": t_loss, "train_accuracy": t_acc,
                "val_loss":   v_loss, "val_accuracy":   v_acc,
            }, step=epoch)

            if v_acc > best_val_acc:
                best_val_acc = v_acc

            print(f"  Epoch {epoch+1:2d}: train_loss={t_loss:.4f}  val_acc={v_acc:.1f}%")

        mlflow.log_metric("best_val_accuracy", best_val_acc)

    w.close()


=== Inicialización: he ===
  Epoch  1: train_loss=2.3180  val_acc=42.0%
  Epoch  2: train_loss=2.0171  val_acc=39.2%
  Epoch  3: train_loss=1.8040  val_acc=39.2%
  Epoch  4: train_loss=1.6774  val_acc=42.5%
  Epoch  5: train_loss=1.6129  val_acc=47.0%
  Epoch  6: train_loss=1.5290  val_acc=45.3%
  Epoch  7: train_loss=1.4910  val_acc=50.8%
  Epoch  8: train_loss=1.4749  val_acc=51.4%
  Epoch  9: train_loss=1.4921  val_acc=54.1%
  Epoch 10: train_loss=1.3884  val_acc=53.6%

=== Inicialización: xavier ===
  Epoch  1: train_loss=2.3873  val_acc=40.9%
  Epoch  2: train_loss=1.9418  val_acc=42.5%
  Epoch  3: train_loss=1.7169  val_acc=43.6%
  Epoch  4: train_loss=1.6189  val_acc=47.0%
  Epoch  5: train_loss=1.5806  val_acc=48.6%
  Epoch  6: train_loss=1.4635  val_acc=50.3%
  Epoch  7: train_loss=1.4542  val_acc=53.6%
  Epoch  8: train_loss=1.3756  val_acc=51.9%
  Epoch  9: train_loss=1.3244  val_acc=48.6%
  Epoch 10: train_loss=1.3485  val_acc=55.8%

=== Inicialización: uniform ===
  Epoch

In [21]:
# !tensorboard --logdir=runs/mlp_experimento_1
